# Ticket T-104: Preprocessing and Data Cleaning Module

This notebook demonstrates and validates the data cleaning and preprocessing pipeline implemented in `src/preprocessing.py` for ticket **T-104**.

### Acceptance Criteria Verified:
1. **Pure & Modular Cleaning Rules**: Pure functions (`calculate_trip_duration`, `filter_outliers`, `drop_banned_columns`, `perform_temporal_split`).
2. **Feature Contract & Leakage Prevention**: Dropping all banned post-trip columns in a single explicit step.
3. **Strict Temporal Split**: Train on first ~3 weeks of May 2022 (`< 2022-05-23`), Test on the final week (`>= 2022-05-23`).
4. **Logged Cleaning Costs**: Explicit row counts dropped per rule.
5. **Parquet Export**: Output saved to `dataset/train_cleaned.parquet` and `dataset/test_cleaned.parquet`.

In [1]:
import os
import sys
from pathlib import Path


In [2]:
# Ensure project root directory is in sys.path when running from notebooks/ directory
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.config import (
    RAW_DATA_PATH,
    TRAIN_CLEANED_PATH,
    TEST_CLEANED_PATH,
    ALLOWED_FEATURES,
    BANNED_COLUMNS,
    TARGET_COLUMNS,
    TEMPORAL_SPLIT_DATE
)
from src.preprocessing import clean_and_preprocess_dataset, drop_banned_columns

## 1. Run Main Preprocessing Pipeline

In [3]:
# Execute preprocessing pipeline
train_df, test_df, stats = clean_and_preprocess_dataset(RAW_DATA_PATH, save_output=True)

2026-08-04 11:10:34,917 - INFO - Loading raw dataset from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/yellow_tripdata_2022-05.parquet...
2026-08-04 11:10:37,611 - INFO - Initial raw rows: 3,588,295
2026-08-04 11:10:37,611 - INFO -   - Dropped by out_of_bound_timestamps: 143
2026-08-04 11:10:37,612 - INFO -   - Dropped by invalid_durations: 44,003
2026-08-04 11:10:37,613 - INFO -   - Dropped by invalid_fares: 17,857
2026-08-04 11:10:37,614 - INFO -   - Dropped by invalid_trip_distances: 19,493
2026-08-04 11:10:37,614 - INFO -   - Dropped by invalid_passenger_counts: 196,445
2026-08-04 11:10:37,615 - INFO -   - Dropped by invalid_ratecodes: 7,863
2026-08-04 11:10:37,615 - INFO -   - Dropped by invalid_location_ids: 0
2026-08-04 11:10:37,616 - INFO - Total rows dropped: 285,804 (7.959999999999994%)
2026-08-04 11:10:37,616 - INFO - Final clean rows: 3,302,491 (92.04%)
2026-08-04 11:10:37,810 - INFO - Temporal Split at 2022-05-23 00:00:00:
2026-08-04 11:10:37,811 -

## 2. Cleaning Cost Breakdown

In [4]:
stats_df = pd.DataFrame(list(stats.items()), columns=["Cleaning Rule / Metric", "Value"])
stats_df

,Cleaning Rule / Metric,Value
0,out_of_bound_timestamps,143.00
1,invalid_durations,44003.00
2,invalid_fares,17857.00
3,invalid_trip_distances,19493.00
4,invalid_passenger_counts,196445.00
5,invalid_ratecodes,7863.00
6,invalid_location_ids,0.00
7,total_initial_rows,3588295.00
8,total_final_rows,3302491.00
9,total_dropped_rows,285804.00


## 3. Leakage Prevention & Feature Contract Verification

In [5]:
print("Train Columns:", train_df.columns.tolist())
print("Test Columns:", test_df.columns.tolist())

# Assert no banned columns survive
for banned in BANNED_COLUMNS:
    assert banned not in train_df.columns, f"Leakage Error: {banned} found in Train!"
    assert banned not in test_df.columns, f"Leakage Error: {banned} found in Test!"

print("\nSUCCESS: All banned post-trip leakage columns successfully removed!")

Train Columns: ['tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'RatecodeID', 'trip_distance', 'VendorID', 'fare_amount', 'duration_minutes']
Test Columns: ['tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'RatecodeID', 'trip_distance', 'VendorID', 'fare_amount', 'duration_minutes']

SUCCESS: All banned post-trip leakage columns successfully removed!


## 4. Temporal Split Boundary Verification

In [6]:
print(f"Split Date Boundary: {TEMPORAL_SPLIT_DATE}")
print(f"Train Min Pickup: {train_df['tpep_pickup_datetime'].min()}")
print(f"Train Max Pickup: {train_df['tpep_pickup_datetime'].max()}")
print(f"Test Min Pickup: {test_df['tpep_pickup_datetime'].min()}")
print(f"Test Max Pickup: {test_df['tpep_pickup_datetime'].max()}")

assert train_df['tpep_pickup_datetime'].max() < pd.to_datetime(TEMPORAL_SPLIT_DATE)
assert test_df['tpep_pickup_datetime'].min() >= pd.to_datetime(TEMPORAL_SPLIT_DATE)
print("\nSUCCESS: Strict temporal split verified!")

Split Date Boundary: 2022-05-23 00:00:00
Train Min Pickup: 2022-05-01 00:00:01
Train Max Pickup: 2022-05-22 23:59:59
Test Min Pickup: 2022-05-23 00:00:02
Test Max Pickup: 2022-05-31 23:59:59

SUCCESS: Strict temporal split verified!
